In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

In [ ]:
data_dir = Path("data/farneback/subj1_cam1")
video_path = data_dir / "28-MAY-2025_00h04m33.969s.mp4"

# ROI: x, y, width, height (adjust to your video)
x, y, w, h = 200, 100, 120, 80

cap = cv2.VideoCapture(str(video_path))
fps = cap.get(cv2.CAP_PROP_FPS)

ret, frame = cap.read()
if not ret:
    raise RuntimeError("Cannot read video")

In [ ]:
# 10m 38s on Mac for 5m-long video
prev = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
# prev = prev[y:y+h, x:x+w]

times = []
intensities = []
frame_idx = 1

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    # curr = gray[y:y+h, x:x+w]

    flow = cv2.calcOpticalFlowFarneback(
        prev, gray, None,
        pyr_scale=0.5,
        levels=3,
        winsize=15,
        iterations=3,
        poly_n=5,
        poly_sigma=1.2,
        flags=0,
    )

    magnitude = np.linalg.norm(flow, axis=2)

    # Ignore tiny displacements likely caused by noise
    magnitude[magnitude < 0.3] = 0

    # Mean motion intensity, in pixels per second
    intensity = magnitude.mean() * fps

    times.append(frame_idx / fps)
    intensities.append(intensity)

    prev = gray # curr
    frame_idx += 1

cap.release()

In [ ]:
plt.plot(times, intensities)
plt.xlabel("Time (s)")
plt.ylabel("Motion intensity (pixels/s)")
plt.tight_layout()
plt.savefig(video_path.parent / f"{video_path.stem}_farneback_intensity.pdf")